In [ ]:
eval "$(~/miniconda3/bin/conda shell.bash hook)"
conda activate pertpy

export LD_LIBRARY_PATH=/home/k-makino/miniconda3/envs/pertpy/lib:$LD_LIBRARY_PATH
export R_HOME=/home/k-makino/miniconda3/envs/pertpy/lib/R
export LD_LIBRARY_PATH=/home/k-makino/miniconda3/envs/pertpy/lib/R/lib:/home/k-makino/miniconda3/envs/pertpy/lib:${LD_LIBRARY_PATH}
export PATH=/home/k-makino/miniconda3/envs/pertpy/bin:${PATH}

# Milo - KNN based differential abundance analysis

## Setup

In [ ]:
import os
os.chdir(path_to_wd)

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import matplotlib
import matplotlib.pyplot as plt
import pertpy as pt
import scanpy as sc
import scvi
import anndata
import pandas as pd
import numpy as np
import pycats
from pandas.api.types import CategoricalDtype

milo = pt.tl.Milo()

sc.settings.verbosity = 3

scvi.settings.seed = 1234
plt.rcParams["figure.figsize"] = (7, 7)
plt.rcParams['font.family']=['Dejavu serif']
plt.rcParams['figure.dpi'] = 100
plt.rcParams['pdf.fonttype'] = 'truetype'

In [ ]:
adata_full = sc.read("Reproducibility/Data/UC_DOGMA_RNA_ATAC_ADT_after_QC.h5ad")

from pandas.api.types import CategoricalDtype
STAGE2 = {
    "Early_L": "NMI", 
    "Early_H": "NMI",
    "Advanced": "MI",
    "post_BCG": "post_BCG"
}
adata_full.obs["STAGE2"] = adata_full.obs["STAGE"].replace(STAGE2)
adata_full.obs["STAGE2"] = adata_full.obs["STAGE2"].astype('category')

In [ ]:
dir_path = "Reproducibility/Results/Milo/output/"
os.makedirs(dir_path, exist_ok = True)

## Lineage

In [ ]:
def preprocess(adata, lineage, k=50):
    adata = adata_full[adata_full.obs['lineage']==lineage].copy()
    ###############################
    tmp_path = f"Reproducibility/Data/embeddings/UC_DOGMA_{lineage}_latent_df.txt"
    latent_df = pd.read_csv(tmp_path,  sep='\t', index_col = 0)
    latent_df = latent_df.loc[adata.obs_names, : ].values 
    adata.obsm["MultiVI_latent"] = latent_df
    ###############################
    tmp_path = f"Reproducibility/Data/embeddings/UC_DOGMA_{lineage}_UMAP_df.txt"
    UMAP_df = pd.read_csv(tmp_path,  sep='\t', index_col = 0)
    adata.obsm["X_umap"] = UMAP_df.values
    sc.pp.neighbors(adata, use_rep="MultiVI_latent")
    if lineage == 'CD4_T':  
        sig_df = pd.read_csv("Reproducibility/Results/VISIONR/UC_DOGMA_CD4_T_signature_score_literature.txt",sep='\t', index_col=0)
        adata.obs = pd.merge(left=adata.obs, right=sig_df.loc[adata.obs_names,:], how='left', left_index=True, right_index=True)
    ###############################
    ## Exclude BCG samples
    adata = adata[adata.obs["STAGE"] != "post_BCG"].copy()
    ## Initialize object for Milo analysis
    milo = pt.tl.Milo()
    mdata = milo.load(adata)
    ################################
    ## Build KNN graph
    sc.pp.neighbors(mdata["rna"], use_rep="MultiVI_latent", n_neighbors=k)
    milo.make_nhoods(mdata["rna"], prop=0.1)
    nhood_size = np.array(mdata["rna"].obsm["nhoods"].sum(0)).ravel()
    plt.hist(nhood_size, bins=30)
    plt.xlabel("# cells in nhood")
    plt.ylabel("# nhoods")
    plt.show()
    ################################
    mdata = milo.count_nhoods(mdata, sample_col="sample")
    return mdata

In [ ]:
def milo_analysis_STAGE_ctrl_Early_L(mdata, lineage, anno_col = "celltype"):
    severity_order = ["Early_L", "Early_H", "Advanced"]
    mdata["rna"].obs["STAGE"] = mdata["rna"].obs["STAGE"].astype("category")
    mdata["rna"].obs["STAGE"] = mdata["rna"].obs["STAGE"].cat.reorder_categories(severity_order)
    ######################
    for target in ["Early_H", "Advanced"]:
        model_contrasts = f"STAGE{target}"
        milo.da_nhoods(mdata, design="~Organ+STAGE", model_contrasts=model_contrasts)
        milo.build_nhood_graph(mdata)
        milo.annotate_nhoods(mdata, anno_col=anno_col)
        ##################
        old_figsize = plt.rcParams["figure.figsize"]
        plt.rcParams["figure.figsize"] = [10, 5]
        plt.subplot(1, 2, 1)
        plt.hist(mdata["milo"].var.PValue, bins=50)
        plt.xlabel("P-Vals")
        plt.subplot(1, 2, 2)
        plt.plot(mdata["milo"].var.logFC, -np.log10(mdata["milo"].var.FDR), ".")
        plt.xlabel("log-Fold Change")
        plt.ylabel("- log10(Spatial FDR)")
        plt.tight_layout()
        plt.rcParams["figure.figsize"] = old_figsize
        plt.show()
        ##################
        milo.plot_da_beeswarm(mdata, alpha=0.1, show=False, return_fig=False)
        plt.show()
        ##################
        sc.set_figure_params(figsize=(3, 3)) 
        sc.set_figure_params(vector_friendly=True, dpi_save=100)
        milo.plot_nhood_graph(
            mdata,
            alpha=0.1,  ## SpatialFDR level (1%)
            min_size=1,  ## Size of smallest dot
            title=f'{target} vs Early_L',
            color_map=None, 
            palette=None, 
            show=False,
        )
        plt.show()
        ###################
        mdata["milo"].var.to_csv(f"{dir_path}Milo_{lineage}_design_Organ_STAGE_contrasts_{target}_vs_Early_L_result.txt", sep = '\t')

In [ ]:
def milo_analysis_STAGE_ctrl_NMI(mdata, lineage, anno_col = "celltype"):
    severity_order = ["NMI", "MI"]
    mdata["rna"].obs["STAGE2"] = mdata["rna"].obs["STAGE2"].astype("category")
    mdata["rna"].obs["STAGE2"] = mdata["rna"].obs["STAGE2"].cat.reorder_categories(severity_order)
    ######################
    for target in ["MI"]:
        model_contrasts = f"STAGE2{target}"
        milo.da_nhoods(mdata, design="~Organ+STAGE2", model_contrasts=model_contrasts)
        milo.build_nhood_graph(mdata)
        milo.annotate_nhoods(mdata, anno_col=anno_col)
        ##################
        old_figsize = plt.rcParams["figure.figsize"]
        plt.rcParams["figure.figsize"] = [10, 5]
        plt.subplot(1, 2, 1)
        plt.hist(mdata["milo"].var.PValue, bins=50)
        plt.xlabel("P-Vals")
        plt.subplot(1, 2, 2)
        plt.plot(mdata["milo"].var.logFC, -np.log10(mdata["milo"].var.FDR), ".")
        plt.xlabel("log-Fold Change")
        plt.ylabel("- log10(Spatial FDR)")
        plt.tight_layout()
        plt.rcParams["figure.figsize"] = old_figsize
        plt.show()
        ##################
        milo.plot_da_beeswarm(mdata, alpha=0.1, show=False, return_fig=False)
        plt.show()
        ##################
        sc.set_figure_params(figsize=(3, 3)) 
        sc.set_figure_params(vector_friendly=True, dpi_save=100)
        milo.plot_nhood_graph(
            mdata,
            alpha=0.1,  ## SpatialFDR level (1%)
            min_size=1,  ## Size of smallest dot
            title=f'{target} vs NMI',
            color_map=None, 
            palette=None, 
            show=False,
        )
        plt.show()
        ###################
        mdata["milo"].var.to_csv(f"{dir_path}Milo_{lineage}_design_Organ_STAGE2_contrasts_{target}_vs_NMI_result.txt", sep = '\t')

In [ ]:
def milo_analysis_STAGE_ctrl_NMI_CD4(mdata, lineage, anno_col = "celltype"):
    severity_order = ["NMI", "MI"]
    mdata["rna"].obs["STAGE2"] = mdata["rna"].obs["STAGE2"].astype("category")
    mdata["rna"].obs["STAGE2"] = mdata["rna"].obs["STAGE2"].cat.reorder_categories(severity_order)
    ######################
    for target in ["MI"]:
        model_contrasts = f"STAGE2{target}"
        milo.da_nhoods(mdata, design="~Organ+STAGE2", model_contrasts=model_contrasts)
        milo.build_nhood_graph(mdata)
        milo.annotate_nhoods(mdata, anno_col=anno_col)
        milo.annotate_nhoods_continuous(mdata, anno_col="IFN_signaling")
        milo.annotate_nhoods_continuous(mdata, anno_col="Glycolysis")
        ##################
        sc.set_figure_params(figsize=(3, 3)) 
        sc.set_figure_params(vector_friendly=True, dpi_save=100)
        milo.plot_nhood_graph(
            mdata,
            alpha=0.1,  ## SpatialFDR level (1%)
            min_size=1,  ## Size of smallest dot
            title=f'{target} vs NMI',
            color_map=None, 
            palette=None, 
            show=False,
        )
        plt.savefig("Reproducibility/Results/Plots/CD4_T/Figure4G.pdf", bbox_inches='tight')
        plt.close()
        ###################
        mdata["milo"].var.to_csv(f"{dir_path}Milo_{lineage}_design_Organ_STAGE2_contrasts_{target}_vs_NMI_result.txt", sep = '\t')

In [ ]:
for lineage in ['CD8_T_NK_ILC','Myeloid','B','MSC']:
    mdata = preprocess(adata = adata_full, lineage = lineage, k=50)
    milo_analysis_STAGE_ctrl_Early_L(mdata = mdata, lineage = lineage, anno_col = "celltype")
    milo_analysis_STAGE_ctrl_NMI(mdata = mdata, lineage = lineage, anno_col = "celltype")

In [ ]:
for lineage in ['CD4_T']:
    mdata = preprocess(adata = adata_full, lineage = lineage, k=50)
    milo_analysis_STAGE_ctrl_Early_L(mdata = mdata, lineage = lineage, anno_col = "celltype")
    milo_analysis_STAGE_ctrl_NMI_CD4(mdata = mdata, lineage = lineage, anno_col = "celltype")

### Effector Tregs

In [ ]:
milo_res = mdata['milo'].var.copy()

## Select nhoods of interest
eTreg_res = milo_res[milo_res.nhood_annotation.isin(['Treg_effector'])]
MI_enr_eTreg_nhoods = eTreg_res.index[(eTreg_res.SpatialFDR < 0.1) & (eTreg_res.logFC > 0)]
NMI_enr_eTreg_nhoods = eTreg_res.index[(eTreg_res.SpatialFDR < 0.1) & (eTreg_res.logFC < 0)]
other_eTreg_nhoods = eTreg_res.index[~(eTreg_res.SpatialFDR < 0.1)]

In [ ]:
# Extract relevant data from `milo`
milo_obs = mdata['milo'].obs
milo_var = mdata['milo'].var
milo_X = mdata['milo'].X  # Transpose the data matrix for correct orientation

# Create a new AnnData object
nhood_adata = anndata.AnnData(
    X=milo_X.T if isinstance(milo_X, np.ndarray) else milo_X.transpose(),
    obs=milo_var,
    var=milo_obs
)

nhood_adata.obs['nhood_groups'] = np.nan
nhood_adata.obs.loc[MI_enr_eTreg_nhoods, 'nhood_groups'] = 'MI_enr'
nhood_adata.obs.loc[NMI_enr_eTreg_nhoods, 'nhood_groups'] = 'NMI_enr'
nhood_adata.obs.loc[other_eTreg_nhoods, 'nhood_groups'] = 'other_eTregs'
mdata['rna'].uns['nhood_adata'] = nhood_adata.copy()

In [ ]:
## find_nhood_group_markers
adata = mdata['rna'][mdata['rna'].obs['celltype'] == 'Treg_effector'].copy()

# Set some parameter
min_n_nhoods = 3
n_hvgs = 7500
test_group='MI_enr'
ctrl_group='NMI_enr'
# confounders_obs = ['Organ']

## Find cells in nhoods of interest
import diff2atlas
groups = adata.uns['nhood_adata'].obs['nhood_groups'].dropna().unique()
for g in groups:
    nhoods_oi = adata.uns['nhood_adata'].obs_names[adata.uns['nhood_adata'].obs['nhood_groups'] == g]
    diff2atlas.utils.get_cells_in_nhoods(adata, nhoods_oi)
    adata.obs[f'in_nhoods_{g}'] = adata.obs['in_nhoods'].copy()

## Find most representative group (cell belongs to mostly to neighbourhoods of that group)
adata.obs['nhood_groups'] = np.nan

In [ ]:
# Define a function to apply the rules
def assign_nhood_group(row):
    # Get the column names and their values
    values = {
        'in_nhoods_NMI_enr': row['in_nhoods_NMI_enr'],
        'in_nhoods_MI_enr': row['in_nhoods_MI_enr'],
        'in_nhoods_other_eTregs': row['in_nhoods_other_eTregs']
    }
    # Find the max value and corresponding column
    max_col = max(values, key=values.get)
    max_val = values[max_col]
    # Apply conditional rules
    if (values['in_nhoods_NMI_enr'] == values['in_nhoods_other_eTregs'] > values['in_nhoods_MI_enr']):
        return 'in_nhoods_NMI_enr'
    elif (values['in_nhoods_MI_enr'] == values['in_nhoods_other_eTregs'] > values['in_nhoods_NMI_enr']):
        return 'in_nhoods_MI_enr'
    else:
        return max_col

# Apply the function to the dataframe
adata.obs['nhood_groups'] = adata.obs.apply(assign_nhood_group, axis=1)

adata.obs.to_csv('Reproducibility/Results/Milo/output/Milo_Treg_effector_metadata.txt', sep='\t')

## BCG paired (immune)

In [ ]:
lineage = 'BCG'

tmp_path = f"Reproducibility/Data/embeddings/UC_DOGMA_{lineage}_latent_df.txt"
latent_df = pd.read_csv(tmp_path,  sep='\t', index_col = 0)

tmp_path = f"Reproducibility/Data/embeddings/UC_DOGMA_{lineage}_UMAP_df.txt"
UMAP_df = pd.read_csv(tmp_path,  sep='\t', index_col = 0)

adata = adata_full[latent_df.index,:].copy()
adata.obsm["MultiVI_latent"] = latent_df.values
adata.obsm["X_umap"] = UMAP_df.values

#--------------------
BCG =  {"pre": ["BC_011",'BC_023','BC_033','BC_037'],
        "post": ["BC_039",'BC_044','BC_048','BC_047']
        }

cat_type = CategoricalDtype(categories=["pre", "post"], ordered=True)
adata.obs['BCG'] = adata.obs["sample"].astype('category')
adata.obs['BCG'] = pycats.cat_collapse(adata.obs["BCG"], BCG)
adata.obs['BCG'] = adata.obs['BCG'].astype(cat_type)

#--------------------
pair = {
        "P1" : ["BC_011","BC_039"],
        "P2" : ["BC_023","BC_044"],
        "P3" : ["BC_033","BC_048"],
        "P4" : ["BC_037","BC_047"]
        }

flat_pair = {v: k for k, values in pair.items() for v in values}
adata.obs["patient"] = adata.obs["sample"].map(flat_pair)
adata.obs["patient"] = adata.obs["patient"].astype('category')

#--------------------

adata = adata[adata.obs['BCG'].isin(['pre','post'])]

In [ ]:
# Replace 'TAM_FOLR2' and 'TAM_TREM2' with 'TAM'
adata.obs['celltype'] = adata.obs['celltype'].replace(
    {'TAM_FOLR2': 'TAM', 'TAM_TREM2': 'TAM'}
)
adata.obs['celltype'] = adata.obs['celltype'].replace(
    {'CD8_Tex_1': 'CD8_Tex', 'CD8_Tex_2': 'CD8_Tex'}
)
# Confirm the changes
print(adata.obs['celltype'].unique())

In [ ]:
## Initialize object for Milo analysis
milo = pt.tl.Milo()
mdata = milo.load(adata)

## Build KNN graph
sc.pp.neighbors(mdata["rna"], use_rep="MultiVI_latent", n_neighbors=20)
milo.make_nhoods(mdata["rna"], prop=0.1)
nhood_size = np.array(mdata["rna"].obsm["nhoods"].sum(0)).ravel()
plt.hist(nhood_size, bins=30)
plt.xlabel("# cells in nhood")
plt.ylabel("# nhoods")
plt.show()

mdata = milo.count_nhoods(mdata, sample_col="sample")

In [ ]:
severity_order = ["pre", "post"]
mdata["rna"].obs["BCG"] = mdata["rna"].obs["BCG"].astype("category")
mdata["rna"].obs["BCG"] = mdata["rna"].obs["BCG"].cat.reorder_categories(severity_order)
######################
target = "post"
#model_contrasts = f"BCG{target}"
milo.da_nhoods(mdata, design="~patient+BCG")
milo.build_nhood_graph(mdata)
milo.annotate_nhoods(mdata, anno_col='celltype')
##################
old_figsize = plt.rcParams["figure.figsize"]
plt.rcParams["figure.figsize"] = [10, 5]
plt.subplot(1, 2, 1)
plt.hist(mdata["milo"].var.PValue, bins=50)
plt.xlabel("P-Vals")
plt.subplot(1, 2, 2)
plt.plot(mdata["milo"].var.logFC, -np.log10(mdata["milo"].var.FDR), ".")
plt.xlabel("log-Fold Change")
plt.ylabel("- log10(Spatial FDR)")
plt.tight_layout()
plt.rcParams["figure.figsize"] = old_figsize
plt.show()
##################
milo.plot_da_beeswarm(mdata, alpha=0.1, show=False, return_fig=False)
plt.show()
##################
sc.set_figure_params(figsize=(3, 3)) 
sc.set_figure_params(vector_friendly=True, dpi_save=100)
milo.plot_nhood_graph(
    mdata,
    alpha=0.1,  ## SpatialFDR level (1%)
    min_size=1,  ## Size of smallest dot
    title=f'{target} vs pre',
    color_map=None, 
    palette=None, 
    show=False,
)
plt.show()
###################
mdata["milo"].var.to_csv(f"{dir_path}Milo_{lineage}_design_patient_BCG_contrasts_{target}_vs_pre_result_TAM_merge_ver.txt", sep = '\t')